# The Price Is Right — with Qwen2.5-7B + QLoRA

### Strategy :
1. **Larger base model**: `Qwen/Qwen2.5-7B-Instruct` (7B vs 3B parameters)
2. **Higher LoRA rank**: r=64, alpha=128 (vs r=32)
3. **More LoRA target modules**: all linear layers
4. **2 training epochs** over the full dataset
5. **Improved inference**: greedy + median ensemble of 3 samples

> Run on T4 16GB (recommended)


In [ ]:
!pip install -q --upgrade bitsandbytes trl peft transformers datasets accelerate
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

In [ ]:
import os
import re
import math
import torch
import statistics
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer, SFTConfig
from datetime import datetime
from util import evaluate

In [ ]:
# ── Constants ──────────────────────────────────────────────────────────────
BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"   # larger & smarter than Llama-3.2-3B
HF_USER      = "iJoshy"                        # ← change to your HF username
PROJECT_NAME = "price-qwen25-7b"
RUN_NAME     = datetime.now().strftime("%Y-%m-%d_%H.%M.%S")
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_NAME}-{RUN_NAME}"

DATASET_NAME = "ed-donner/items_prompts_full"

# ── LoRA hyper-parameters (higher rank → more capacity) ──────────────────
LORA_R           = 64      # baseline used 32
LORA_ALPHA       = 128     # 2× rank is a solid rule of thumb
LORA_DROPOUT     = 0.05
LORA_TARGET_MODS = "all-linear"   # adapt every linear layer

# ── Training hyper-parameters ─────────────────────────────────────────────
NUM_EPOCHS        = 2
LEARNING_RATE     = 2e-4
PER_DEVICE_BATCH  = 4
GRAD_ACCUM        = 4       # effective batch = 16
MAX_SEQ_LENGTH    = 180
WARMUP_RATIO      = 0.05

capability = torch.cuda.get_device_capability()
USE_BF16   = capability[0] >= 8
print(f"BF16: {USE_BF16} | Device capability: {capability}")

In [ ]:
# ── HuggingFace login ──────────────────────────────────────────────────────
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────
dataset = load_dataset(DATASET_NAME)
train   = dataset["train"]
test    = dataset["test"]
print(f"Train: {len(train):,}  |  Test: {len(test):,}")
print(train[0])

In [ ]:
# ── Formatting function ────────────────────────────────────────────────────
# Concatenate prompt + completion so SFTTrainer can train on the full text
# but we only compute loss on the completion token(s).
def format_example(example):
    return {"text": example["prompt"] + example["completion"]}

In [ ]:
# ── 4-bit quantization ─────────────────────────────────────────────────────
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_quant_type="nf4",
)

# ── Load tokenizer ─────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.padding_side  = "right"

# ── Load base model ────────────────────────────────────────────────────────
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
print(f"Base model memory: {base_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
# ── LoRA configuration ─────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODS,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in base_model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# ── SFT training arguments ─────────────────────────────────────────────────
OUTPUT_DIR = f"./results/{PROJECT_NAME}-{RUN_NAME}"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    save_steps=0,
    logging_steps=25,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    max_grad_norm=0.3,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    report_to="none",
    dataset_text_field="text",
    packing=True,
)

In [ ]:
# ── Format training data ───────────────────────────────────────────────────
train_formatted = train.map(format_example)

# ── Trainer ────────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_formatted,
    peft_config=lora_config,
    args=sft_config,
)

trainer.train()

In [ ]:
# ── Save & push adapter to HuggingFace Hub ─────────────────────────────────
trainer.model.push_to_hub(HUB_MODEL_NAME)
tokenizer.push_to_hub(HUB_MODEL_NAME)
print(f"\nModel pushed to: https://huggingface.co/{HUB_MODEL_NAME}")

---
## Evaluation

If you're loading a previously pushed model, set `HUB_MODEL_NAME` and run from here.

In [ ]:
# ── Reload fine-tuned model for evaluation ─────────────────────────────────
# (Skip if you just trained above — trainer.model already has the LoRA weights)

# Uncomment to load from Hub:
# base_model = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL, quantization_config=quant_config, device_map="auto", trust_remote_code=True
# )
# fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)

# Using the trainer model directly:
fine_tuned_model = trainer.model
fine_tuned_model.eval()
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
# ── Improved inference: median of 3 independent samples ───────────────────
def parse_price(text: str) -> float:
    text = text.replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", text)
    return float(m.group()) if m else 0.0


def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    prompt_len = inputs["input_ids"].shape[1]

    # Three stochastic samples → take median to reduce variance
    samples = []
    with torch.no_grad():
        for _ in range(3):
            out = fine_tuned_model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=True,
                temperature=0.3,
                top_p=0.9,
            )
            gen_ids = out[0, prompt_len:]
            raw = tokenizer.decode(gen_ids, skip_special_tokens=True)
            samples.append(parse_price(raw))

    return statistics.median(samples)

In [ ]:
# ── THE MOMENT OF TRUTH ────────────────────────────────────────────────────
set_seed(42)
evaluate(model_predict, test)